In [34]:
from functools import partial
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import bayesflow.diagnostics as diag
from bayesflow.amortizers import AmortizedPosterior
from bayesflow.networks import InvertibleNetwork
from bayesflow.simulation import GenerativeModel, Prior, Simulator
from bayesflow.trainers import Trainer
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels as sm
import statsmodels.api as sm
import copy
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mvn, gamma as xgamma, norm,multivariate_normal
from scipy.special import gamma, factorial
from timeit import default_timer as timer
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv2D, MaxPooling2D, TimeDistributed, Flatten, Dense
rng = np.random.default_rng(123)

# Configuración para permitir el uso de toda la GPU disponible
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    # Hacer visibles todas las GPUs disponibles
    tf.config.set_visible_devices(physical_devices, 'GPU')
    print(f"Se detectaron {len(physical_devices)} GPU(s) y se han configurado para su uso.")
    # Configurar para permitir el crecimiento dinámico de la memoria de cada GPU
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
else:
    print("No se detectaron GPUs.")

No se detectaron GPUs.


In [35]:

n_sitios = 10
n_obs = 120
n = n_sitios * n_obs

# Coordenadas de cada sitio
lon_sitios = rng.uniform(-1, -0.5, size=n_sitios)
lat_sitios = rng.uniform(0.5, 1, size=n_sitios)
oscilacion_base = np.sin(np.linspace(0, 2*np.pi, n_obs))

X = pd.DataFrame({
    "intercepto": np.ones(n),
    "dummy": rng.integers(0, 2, size=n),
    "lon": np.repeat(lon_sitios, n_obs),
    "lat": np.repeat(lat_sitios, n_obs),
    "oscilacion": np.tile(oscilacion_base, n_sitios)
})

# Betas verdaderos entre -2 y 2
beta = [3, 0.2, -0.05, 0.05, 2]

# Datos sintéticos ("reales")
Y = np.exp(X.values @ beta) #+ rng.normal(0, 0.1, size=n))

print("Betas verdaderos:")
print(beta)
print(Y)

Betas verdaderos:
[3, 0.2, -0.05, 0.05, 2]
[26.33148746 23.9583321  26.61760036 ... 17.37269305 19.30098468
 21.4496174 ]


In [36]:
def model_prior():
    # betas = rng.uniform(low=-10, high=10, size=(n_sim, n_betas))
    betas = rng.normal(loc=0, scale=5, size=(5))
    return betas

parametros= [ r"$\beta_0$", r"$\beta_1$",r"$\beta_2$", r"$\beta_3$",r"$\beta_4$"]
prior = Prior(prior_fun=model_prior, param_names=parametros)
prior_means, prior_stds = prior.estimate_means_and_stds()
print("Medias del prior:", prior_means)

Medias del prior: [[ 0.19522748  0.12696365  0.15659885 -0.13351551 -0.23339729]]


In [37]:
def proceso_D8(params):
    Y = np.exp(X.values @ params) #+ rng.normal(0, 0.1, size=n))
    return(Y)
simulator_D8 = Simulator(simulator_fun=partial(proceso_D8))
model_D8 = GenerativeModel(prior, simulator_D8, name="simulador_proceso")
data_D8 = model_D8(batch_size=1)

print('Inicia simulacion')
myobj=datetime.now()
print(myobj)
simul_previa_D8 = model_D8(batch_size=10000)
print('Termina simulacion')
myobj=datetime.now()
print(myobj)





INFO:root:Performing 2 pilot runs with the simulador_proceso model...
INFO:root:Shape of parameter batch after 2 pilot simulations: (batch_size = 2, 5)
INFO:root:Shape of simulation batch after 2 pilot simulations: (batch_size = 2, 1200)
INFO:root:No optional prior non-batchable context provided.
INFO:root:No optional prior batchable context provided.
INFO:root:No optional simulation non-batchable context provided.
INFO:root:No optional simulation batchable context provided.


Inicia simulacion
2026-07-06 20:48:28.718893
Termina simulacion
2026-07-06 20:48:29.112576


In [38]:
class CustomLSTM_D8(tf.keras.Model):
    def __init__(self, hidden_size=1000, summary_dim=2000):
        super().__init__()

        self.LSTM = tf.keras.Sequential(
            [
                tf.keras.layers.LSTM(hidden_size, return_sequences=True),
                tf.keras.layers.LSTM(hidden_size, return_sequences=False),
                tf.keras.layers.Dense(hidden_size, activation="relu"),
                tf.keras.layers.Dense(summary_dim, activation="elu"),
            ]
        )

    def call(self, x, **kwargs):
        # Si viene como (batch, 1200), convertir a (batch, 1200, 1)
        if len(x.shape) == 2:
            x = tf.expand_dims(x, axis=-1)

        return self.LSTM(x)

In [58]:

COUPLING_NET_SETTINGS = {
    # "dense_args": dict(units=128, kernel_regularizer=None, activation="relu"),
    "num_dense": 2,
    "dropout_prob": 0.2, "bins" : 32
}

summary_net_D8 = CustomLSTM_D8(hidden_size=128, summary_dim=128)
inference_net_D8 = InvertibleNetwork(num_params=5, num_coupling_layers=4, coupling_settings=COUPLING_NET_SETTINGS,coupling_design='spline')
amortizer_D8 = AmortizedPosterior(inference_net_D8, summary_net_D8)
trainer_D8 = Trainer(amortizer=amortizer_D8, generative_model=model_D8, memory=False)


myobj= datetime.now()
print(myobj)


INFO:root:Performing a consistency check with provided components...
INFO:root:Done.


2026-07-06 21:23:06.517814


In [ ]:
n_epochs = 10
n_batch_size = 32


history_D8 = trainer_D8.train_offline(simulations_dict=simul_previa_D8,epochs=n_epochs,batch_size=n_batch_size,early_stopping=True,validation_sims=128)
valid_sim_data_raw_D8 = model_D8(batch_size=512)
valid_sim_data_D8 = trainer_D8.configurator(valid_sim_data_raw_D8)
posterior_samples_D8 = amortizer_D8.sample(valid_sim_data_D8, n_samples=100)
fig = diag.plot_recovery(posterior_samples_D8, valid_sim_data_D8["parameters"], param_names=parametros, xlabel= "Real",ylabel= "Estimado",n_col=5)


INFO:root:Generated 128 simulations for validation.
Training epoch 1:  17%|█▋        | 53/313 [01:10<04:01,  1.08it/s, Epoch: 1, Batch: 53,Loss: 15.913,W.Decay: 0.206,Avg.Loss: 24.861,Avg.W.Decay: 0.206,LR: 5.00E-04]

In [56]:
valid_sim_data_1 = model_D8(batch_size=1)
x = tf.expand_dims(Y, axis=-1)
valid_sim_data_1['sim_data']=x
valid_sim_data_1_c = trainer_D8.configurator(valid_sim_data_1)
posterior_samples = amortizer_D8.sample(valid_sim_data_1_c, n_samples=1000)

In [57]:
posterior_samples[0].mean(axis=0)

array([ 1.0546299 ,  0.08965761, -0.07054986,  0.7351875 ,  0.5558062 ],
      dtype=float32)

In [ ]:

3, 0.2, -0.05, 0.05, 2
